In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, classification_report
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# STEP 1: Load the dataset (no-pollution version — your final choice)
# ============================================================
data_path = r"C:\Users\Admin\OneDrive\Desktop\FYP_Dataset\master_dataset_features.csv"
df = pd.read_csv(data_path)
df['datetime'] = pd.to_datetime(df['datetime'])

# Reconstruct city_name from one-hot encoded columns
# (master_dataset_features.csv doesn't have a direct city_name column)
city_dummy_cols = [c for c in df.columns if c.startswith('city_')]
df['city_name'] = df[city_dummy_cols].idxmax(axis=1).str.replace('city_', '', regex=False)

df = df.sort_values(['city_name', 'datetime']).reset_index(drop=True)

print("Dataset loaded!")
print(f"Shape: {df.shape}")

# ============================================================
# STEP 2: Build the FUTURE target — visibility ~5 hours ahead
# (midpoint of your proposal's 4-6 hour window)
# ============================================================
FORECAST_HOURS = 5
TOLERANCE_HOURS = 1  # allows matching anywhere in the 4-6 hour window

future_rows = []

for city in df['city_name'].unique():
    city_df = df[df['city_name'] == city].sort_values('datetime').reset_index(drop=True)

    # Create a lookup of this city's own data, shifted back in time,
    # so we can match each row to a reading ~5 hours in its future
    lookup = city_df[['datetime', 'visibility']].copy()
    lookup = lookup.rename(columns={'datetime': 'future_datetime', 'visibility': 'visibility_future'})

    city_df['target_datetime'] = city_df['datetime'] + pd.Timedelta(hours=FORECAST_HOURS)

    # merge_asof finds the closest future reading within tolerance
    city_df = city_df.sort_values('target_datetime')
    lookup = lookup.sort_values('future_datetime')

    matched = pd.merge_asof(
        city_df,
        lookup,
        left_on='target_datetime',
        right_on='future_datetime',
        direction='nearest',
        tolerance=pd.Timedelta(hours=TOLERANCE_HOURS)
    )

    future_rows.append(matched)

df_forecast = pd.concat(future_rows, ignore_index=True)

# Drop rows where no future reading was found within tolerance
# (this naturally happens near the end of the dataset for each city)
before = len(df_forecast)
df_forecast = df_forecast.dropna(subset=['visibility_future']).reset_index(drop=True)
after = len(df_forecast)
print(f"\nRows before dropping unmatched future readings: {before}")
print(f"Rows after: {after} ({before - after} dropped — expected near dataset end per city)")

# ============================================================
# STEP 3: Build X (features) and y (future visibility target)
# ============================================================
# NOTE: 'visibility' (current reading) is now a VALID FEATURE,
# since it's present-moment info being used to predict the future —
# it is no longer the target, so we KEEP it in X this time.
exclude_cols = ['visibility_future', 'datetime', 'target_datetime',
                 'future_datetime', 'time_gap_hours', 'city_name']
feature_cols = [c for c in df_forecast.columns if c not in exclude_cols]

X = df_forecast[feature_cols]
y = df_forecast['visibility_future']

valid_idx = X.dropna().index
X = X.loc[valid_idx]
y = y.loc[valid_idx]

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")

# Time-based split (do NOT shuffle — keep chronological order)
split_index = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

print(f"Training rows: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Testing rows:  {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")

# ============================================================
# STEP 4: Train weighted XGBoost (same weighting strategy as before)
# ============================================================
sample_weights = np.where(y_train < 0.5, 5, 1)  # CRITICAL rows count 5x more

xgb_forecast_model = xgb.XGBRegressor(
    n_estimators=100, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1
)

xgb_forecast_model.fit(X_train, y_train, sample_weight=sample_weights)
print("\nForecast model training complete!")

# ============================================================
# STEP 5: Evaluate — regression AND classification metrics
# ============================================================
pred = xgb_forecast_model.predict(X_test)

r2 = r2_score(y_test, pred)
mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))

print(f"\n=== Forecast Model (5-hour ahead) — Regression ===")
print(f"R²   : {r2:.4f}")
print(f"MAE  : {mae:.4f} km")
print(f"RMSE : {rmse:.4f} km")

def classify_risk(v):
    if v < 0.5: return "CRITICAL"
    elif v < 2.0: return "HIGH"
    elif v < 5.0: return "MODERATE"
    elif v < 10.0: return "LOW"
    else: return "SAFE"

actual_risk = y_test.apply(classify_risk)
predicted_risk = pd.Series(pred, index=y_test.index).apply(classify_risk)

print(f"\n=== Forecast Model (5-hour ahead) — Risk Classification ===")
print(classification_report(actual_risk, predicted_risk,
                             labels=['CRITICAL','HIGH','MODERATE','LOW','SAFE']))

# ============================================================
# STEP 6: Save the model and test set with predictions
# ============================================================
import joblib
joblib.dump(xgb_forecast_model, r"C:\Users\Admin\OneDrive\Desktop\FYP_Dataset\xgb_model_forecast_5hr.pkl")

test_export = df_forecast.loc[X_test.index].copy()
test_export['predicted_visibility'] = pred
test_export.to_csv(r"C:\Users\Admin\OneDrive\Desktop\FYP_Dataset\test_set_forecast_5hr.csv", index=False)

print("\nForecast model and test set saved!")

Dataset loaded!
Shape: (37234, 26)

Rows before dropping unmatched future readings: 37234
Rows after: 36955 (279 dropped — expected near dataset end per city)

X shape: (36952, 23)
y shape: (36952,)
Training rows: 29561 (80.0%)
Testing rows:  7391 (20.0%)

Forecast model training complete!

=== Forecast Model (5-hour ahead) — Regression ===
R²   : -0.0165
MAE  : 1.0954 km
RMSE : 2.9283 km

=== Forecast Model (5-hour ahead) — Risk Classification ===
              precision    recall  f1-score   support

    CRITICAL       0.58      0.51      0.55       580
        HIGH       0.63      0.58      0.60      2106
    MODERATE       0.80      0.86      0.83      4323
         LOW       0.37      0.31      0.34       256
        SAFE       0.01      0.01      0.01       126

    accuracy                           0.72      7391
   macro avg       0.48      0.45      0.46      7391
weighted avg       0.71      0.72      0.71      7391


Forecast model and test set saved!
